In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DateType
from pyspark.sql.functions import col,when,lit,current_date

In [2]:
Spark = SparkSession.builder.appName('SCD TYPE1').getOrCreate()

In [3]:
schema = StructType(
    [
        StructField("Customer_id",IntegerType(),True),
        StructField("Name",StringType(),True),
        StructField("City",StringType(),True),
        StructField("StartDate",StringType(),True),
        StructField("EndDate",StringType(),True),
        StructField("is_active",StringType(),True)
    ]
)

Target_data = [
    (1001, "Argha", "Kolkata", "2023-01-01", None, "Y"),
    (1002, "Rahul", "Delhi", "2023-01-01", None, "Y"),
    (1003, "John", "Mumbai", "2023-01-05", None, "Y"),
    (1004, "Sham", "Chennai", "2023-01-08", None, "Y"),
    (1005, "Champ", "Noida", "2022-05-04", "2025-03-02", "N")
]

define_schema = StructType([
    StructField("Customer_id",IntegerType(),True),
    StructField("Name",StringType(),True),
    StructField("City",StringType(),True)
])

source_data = [
    (1001, "Argha", "Mumbai"),  # changed city
    (1002, "Rahul", "Delhi"),   # no change
    (1003, "John", "pune"),     # changed city
    (1006, "Amit", "Pune")      # new record
]

dim_df = Spark.createDataFrame(Target_data,schema)
src_df = Spark.createDataFrame(source_data,define_schema)
dim_df.show()
src_df.show()

+-----------+-----+-------+----------+----------+---------+
|Customer_id| Name|   City| StartDate|   EndDate|is_active|
+-----------+-----+-------+----------+----------+---------+
|       1001|Argha|Kolkata|2023-01-01|      null|        Y|
|       1002|Rahul|  Delhi|2023-01-01|      null|        Y|
|       1003| John| Mumbai|2023-01-05|      null|        Y|
|       1004| Sham|Chennai|2023-01-08|      null|        Y|
|       1005|Champ|  Noida|2022-05-04|2025-03-02|        N|
+-----------+-----+-------+----------+----------+---------+

+-----------+-----+------+
|Customer_id| Name|  City|
+-----------+-----+------+
|       1001|Argha|Mumbai|
|       1002|Rahul| Delhi|
|       1003| John|  pune|
|       1006| Amit|  Pune|
+-----------+-----+------+



In [4]:
# Active records from target
active_df = dim_df.filter(col("is_active") == "Y")

# Inactive records from target (keep as it is)
inactive_df = dim_df.filter(col("is_active") == "N")

In [5]:
# Join source with active target
joined_df = src_df.alias("src").join(
    active_df.alias("tgt"),
    col("src.Customer_id") == col("tgt.Customer_id"),
    "left"
)
joined_df.show()

+-----------+-----+------+-----------+-----+-------+----------+-------+---------+
|Customer_id| Name|  City|Customer_id| Name|   City| StartDate|EndDate|is_active|
+-----------+-----+------+-----------+-----+-------+----------+-------+---------+
|       1001|Argha|Mumbai|       1001|Argha|Kolkata|2023-01-01|   null|        Y|
|       1002|Rahul| Delhi|       1002|Rahul|  Delhi|2023-01-01|   null|        Y|
|       1003| John|  pune|       1003| John| Mumbai|2023-01-05|   null|        Y|
|       1006| Amit|  Pune|       null| null|   null|      null|   null|     null|
+-----------+-----+------+-----------+-----+-------+----------+-------+---------+



In [6]:
# Records from source (for update + insert)
upsert_df = joined_df.select(
    col("src.Customer_id").alias("Customer_id"),
    col("src.Name").alias("Name"),
    col("src.City").alias("City"),
    lit(current_date().cast("string")).alias("StartDate"),
    lit(None).cast("string").alias("EndDate"),
    lit("Y").alias("is_active")
)
upsert_df.show()

+-----------+-----+------+----------+-------+---------+
|Customer_id| Name|  City| StartDate|EndDate|is_active|
+-----------+-----+------+----------+-------+---------+
|       1001|Argha|Mumbai|2026-04-18|   null|        Y|
|       1002|Rahul| Delhi|2026-04-18|   null|        Y|
|       1003| John|  pune|2026-04-18|   null|        Y|
|       1006| Amit|  Pune|2026-04-18|   null|        Y|
+-----------+-----+------+----------+-------+---------+



In [7]:
# Unchanged active target records
unchanged_df = active_df.join(
    src_df.select("Customer_id"),
    "Customer_id",
    "left_anti"
)
unchanged_df.show()

+-----------+----+-------+----------+-------+---------+
|Customer_id|Name|   City| StartDate|EndDate|is_active|
+-----------+----+-------+----------+-------+---------+
|       1004|Sham|Chennai|2023-01-08|   null|        Y|
+-----------+----+-------+----------+-------+---------+



In [8]:
final_df = unchanged_df.unionByName(upsert_df).unionByName(inactive_df)

final_df.show(truncate=False)

+-----------+-----+-------+----------+----------+---------+
|Customer_id|Name |City   |StartDate |EndDate   |is_active|
+-----------+-----+-------+----------+----------+---------+
|1004       |Sham |Chennai|2023-01-08|null      |Y        |
|1001       |Argha|Mumbai |2026-04-18|null      |Y        |
|1002       |Rahul|Delhi  |2026-04-18|null      |Y        |
|1003       |John |pune   |2026-04-18|null      |Y        |
|1006       |Amit |Pune   |2026-04-18|null      |Y        |
|1005       |Champ|Noida  |2022-05-04|2025-03-02|N        |
+-----------+-----+-------+----------+----------+---------+

